# F1 driver-movement flow chart (2020–2024)

PySpark pipeline + Graphviz rendering.

**Layout:** years down the left, the 10 teams as colored columns on top, each
driver shown by a 3-letter code plus the season **teammate qualifying
head-to-head** (`wins-losses`, the *out-qualify score*). Solid lines = season-
to-season continuity; **dotted lines = a driver returning after a break**.

**Note on teams — RB is not Red Bull.** `Red Bull` is Red Bull Racing; `RB` is
the sister team (Toro Rosso → AlphaTauri → RB). Renamed constructors are
collapsed to one canonical column so there are exactly 10:
`Renault→Alpine`, `Racing Point→Aston Martin`, `AlphaTauri→RB`,
`Alfa Romeo→Sauber`. That is why the RB column is populated for every season
(it absorbs the AlphaTauri entries).


## 1. Spark session & gold tables


In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

# point this at the unzipped 'gold' folder
GOLD  = "gold"
START, END = 2020, 2024

spark = SparkSession.builder.appName("f1-driver-flow").getOrCreate()

drivers = spark.read.parquet(f"{GOLD}/dim_driver")
cons    = spark.read.parquet(f"{GOLD}/dim_constructor")
facts   = (spark.read.parquet(f"{GOLD}/fact_race_result")
                .filter((F.col("year") >= START) & (F.col("year") <= END)))

## 2. Prepare: abbreviations, canonical teams, qualifying H2H, primary team


In [ ]:
# 3-letter abbreviation: official `code`, else first 3 letters of family name
family = F.element_at(F.split(F.col("driver_name"), " "), -1)
abbr = (F.when(F.col("code").isNotNull() & (F.col("code") != "\\N"),
               F.upper(F.substring(F.col("code"), 1, 3)))
         .otherwise(F.upper(F.substring(family, 1, 3))))
drivers = drivers.withColumn("abbr", abbr)

# collapse renamed constructors -> 10 canonical teams
canon = (F.when(F.col("constructor_name") == "Renault",      F.lit("Alpine F1 Team"))
          .when(F.col("constructor_name") == "Racing Point", F.lit("Aston Martin"))
          .when(F.col("constructor_name") == "AlphaTauri",   F.lit("RB F1 Team"))
          .when(F.col("constructor_name") == "Alfa Romeo",   F.lit("Sauber"))
          .otherwise(F.col("constructor_name")))
cons = cons.withColumn("team", canon)

# teammate qualifying head-to-head: per race the two same-constructor cars are
# compared on qualifying_position; the better grid slot scores a win.
q = (facts.filter(F.col("qualifying_position").isNotNull())
          .select("year", "raceId", "constructorId", "driverId", "qualifying_position"))
mate = q.select("raceId", "constructorId",
                F.col("driverId").alias("mate_id"),
                F.col("qualifying_position").alias("mate_q"))
pairs = (q.join(mate, ["raceId", "constructorId"])
          .filter(F.col("driverId") != F.col("mate_id")))
h2h = (pairs.groupBy("year", "driverId")
            .agg(F.sum(F.when(F.col("qualifying_position") < F.col("mate_q"), 1).otherwise(0)).alias("q_wins"),
                 F.sum(F.when(F.col("qualifying_position") > F.col("mate_q"), 1).otherwise(0)).alias("q_losses"))
            .withColumn("quali_h2h", F.concat_ws("-", "q_wins", "q_losses")))

# starts per (year, driver, canonical team) -> primary team = most starts
starts = (facts.join(cons.select("constructorId", "team"), "constructorId")
               .groupBy("year", "driverId", "team")
               .agg(F.count("*").alias("starts")))
w = Window.partitionBy("year", "driverId").orderBy(F.desc("starts"))
primary = starts.withColumn("rk", F.row_number().over(w)).filter(F.col("rk") == 1).drop("rk")

seasons = (primary
    .join(drivers.select("driverId", "driver_name", "abbr"), "driverId")
    .join(h2h.select("year", "driverId", "quali_h2h"), ["year", "driverId"], "left")
    .select("year", "driverId", "abbr", "driver_name", "team", "starts",
            F.coalesce("quali_h2h", F.lit("0-0")).alias("quali_h2h")))
seasons.orderBy("year", "team").show(120, truncate=False)

## 3. Grid (year × team) and movements (continuation vs return)


In [ ]:
grid = (seasons.groupBy("year", "team")
               .agg(F.sort_array(F.collect_list(
                    F.struct(F.desc("starts").alias("s"), "abbr"))).alias("d"))
               .withColumn("drivers", F.expr("transform(d, x -> x.abbr)"))
               .drop("d").orderBy("year", "team"))

wd = Window.partitionBy("driverId").orderBy("year")
moves = (seasons
    .withColumn("prev_year", F.lag("year").over(wd))
    .withColumn("prev_team", F.lag("team").over(wd))
    .filter(F.col("prev_year").isNotNull())
    .withColumn("gap", F.col("year") - F.col("prev_year"))
    .withColumn("link_type", F.when(F.col("gap") == 1, F.lit("continuation"))
                              .otherwise(F.lit("return")))          # gap > 1 -> dotted
    .withColumn("team_changed", F.col("team") != F.col("prev_team"))
    .select("driverId", "abbr", "from_year", "to_year",
            F.col("prev_team").alias("from_team"), F.col("team").alias("to_team"),
            "link_type", "team_changed")
    .withColumnRenamed("prev_year", "from_year"))
# bring to the driver: collect the small result sets to pandas for plotting
pdf_seasons = seasons.toPandas()
pdf_moves   = (seasons.withColumn("prev_year", F.lag("year").over(wd))
                      .withColumn("prev_team", F.lag("team").over(wd))
                      .filter(F.col("prev_year").isNotNull())
                      .select("driverId", "abbr", "prev_year", "year",
                              "prev_team", "team")).toPandas()
grid.withColumn("drivers", F.concat_ws("|", "drivers")).orderBy("year","team").show(120, truncate=False)

## 4. Render the flow chart (Graphviz `neato`, pinned grid layout)


In [ ]:
import subprocess, collections
from IPython.display import Image, display

YEARS = list(range(START, END + 1))
COL_ORDER = ["Red Bull", "RB F1 Team", "Mercedes", "Ferrari", "McLaren",
             "Aston Martin", "Alpine F1 Team", "Williams", "Haas F1 Team", "Sauber"]
COLOR = {"Red Bull":"#1E2A6E","RB F1 Team":"#2B4FC0","Mercedes":"#00A19B",
         "Ferrari":"#D40000","McLaren":"#FF8000","Aston Martin":"#00594F",
         "Alpine F1 Team":"#0090D0","Williams":"#1868DB","Haas F1 Team":"#9A9A9A",
         "Sauber":"#00B000"}
fg = lambda t: "#000000" if t == "Haas F1 Team" else "#FFFFFF"
colx = {t: i for i, t in enumerate(COL_ORDER)}
COLW, ROWH, SUB = 235.0, 175.0, 70.0

seasons_map = {(int(r.year), int(r.driverId)): (r.team, r.abbr, int(r.starts), r.quali_h2h)
               for r in pdf_seasons.itertuples()}

cell = collections.defaultdict(list)
for (y, d), (team, ab, n, sc) in seasons_map.items():
    cell[(y, team)].append((n, d, ab, sc))

dot = ['digraph F1 {', '  graph [bgcolor="#FFFFFF"];',
       '  node [shape=box, style="filled,rounded", fontname="Helvetica", fixedsize=true, width=0.82, height=0.5];',
       '  edge [penwidth=2.2, arrowsize=0.7];']
for t in COL_ORDER:
    dot.append(f'  h_{colx[t]} [pos="{colx[t]*COLW:.1f},{ROWH*0.85:.1f}", label="{t.replace(" F1 Team","")}", '
               f'shape=box, style="filled", fillcolor="{COLOR[t]}", fontcolor="{fg(t)}", '
               f'fontname="Helvetica-Bold", fontsize=12, width=2.0, height=0.45];')
for y in YEARS:
    dot.append(f'  y_{y} [pos="{-COLW*0.85:.1f},{-YEARS.index(y)*ROWH:.1f}", label="{y}", shape=plaintext, '
               f'fontsize=22, fontname="Helvetica-Bold", fontcolor="#222222", fixedsize=false];')
for (y, team), members in cell.items():
    members.sort(reverse=True); k = len(members)
    base = colx[team]*COLW; yy = -YEARS.index(y)*ROWH
    for i, (n, d, ab, sc) in enumerate(members):
        x = base + (i - (k-1)/2.0)*SUB
        label = f'<<B><FONT POINT-SIZE="14">{ab}</FONT></B><BR/><FONT POINT-SIZE="9">{sc}</FONT>>'
        dot.append(f'  n_{y}_{d} [pos="{x:.1f},{yy:.1f}", label={label}, fillcolor="{COLOR[team]}", fontcolor="{fg(team)}"];')

bydriver = collections.defaultdict(list)
for (y, d) in seasons_map: bydriver[d].append(y)
for d, ys in bydriver.items():
    ys.sort()
    for a, b in zip(ys, ys[1:]):
        color = COLOR[seasons_map[(b, d)][0]]
        style = "solid" if b - a == 1 else "dotted"
        extra = ", penwidth=2.6" if style == "dotted" else ""
        dot.append(f'  n_{a}_{d} -> n_{b}_{d} [color="{color}", style={style}{extra}];')
dot.append('}')

open("chart.dot", "w").write("\n".join(dot))
subprocess.run(["neato", "-n2", "-Tpng", "-Gdpi=150", "chart.dot", "-o", "driver_flow.png"], check=True)
display(Image("driver_flow.png"))

**Reading the chart.** A diagonal *solid* line is a normal year-to-year team
switch; a *dotted* line is a driver who sat out one or more seasons and came
back (e.g. Albon, Magnussen, Hülkenberg). A code with no line entering it is a
debut that season; no line leaving it means that was their last season in the
window.
